# Scaled Dot-Product Attention (缩放点积注意力) 实现

缩放点积注意力是Transformer模型的核心组件，通过缩放因子解决了点积值过大的问题。

**公式**: $\text{Attention}(Q, K, V) = \text{softmax}(\frac{QK^T}{\sqrt{d_k}}) V$

## 1. 导入必要的库

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 设置随机种子
np.random.seed(42)

# 设置绘图样式
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 2. 实现Softmax函数

In [ ]:
def softmax(x, axis=-1):
    """Softmax函数实现"""
    # 减去最大值提高数值稳定性
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

# 测试softmax
test_scores = np.array([1.0, 2.0, 3.0, 4.0])
test_weights = softmax(test_scores)
print(f"输入分数: {test_scores}")
print(f"Softmax权重: {test_weights}")
print(f"权重之和: {test_weights.sum():.6f}")

## 3. 实现缩放点积注意力类

In [ ]:
class ScaledDotProductAttention:
    """缩放点积注意力机制实现类"""
    
    def __init__(self, temperature=None):
        """
        初始化
        Args:
            temperature: 温度参数，默认为None时使用sqrt(d_k)
        """
        self.temperature = temperature
    
    def forward(self, Q, K, V, mask=None):
        """
        前向传播
        Args:
            Q: 查询矩阵 (..., seq_len_q, d_k)
            K: 键矩阵 (..., seq_len_k, d_k)
            V: 值矩阵 (..., seq_len_k, d_v)
            mask: 掩码矩阵（可选）
        Returns:
            output: 注意力输出
            attention_weights: 注意力权重
        """
        d_k = K.shape[-1]
        
        # 步骤1: 计算QK^T
        scores = np.matmul(Q, np.swapaxes(K, -2, -1))
        
        # 步骤2: 缩放
        if self.temperature is None:
            scores = scores / np.sqrt(d_k)
        else:
            scores = scores / self.temperature
        
        # 步骤3: 应用mask
        if mask is not None:
            scores = np.where(mask, -1e9, scores)
        
        # 步骤4: Softmax
        attention_weights = softmax(scores, axis=-1)
        
        # 步骤5: 加权求和
        output = np.matmul(attention_weights, V)
        
        return output, attention_weights

print("✓ ScaledDotProductAttention类定义完成")

## 4. 简化版函数实现

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """缩放点积注意力的简化函数实现"""
    d_k = K.shape[-1]
    
    # QK^T / sqrt(d_k)
    scores = np.matmul(Q, np.swapaxes(K, -2, -1)) / np.sqrt(d_k)
    
    # 应用mask
    if mask is not None:
        scores = np.where(mask, -1e9, scores)
    
    # Softmax
    attention_weights = softmax(scores, axis=-1)
    
    # 加权求和
    output = np.matmul(attention_weights, V)
    
    return output, attention_weights

print("✓ 简化版函数定义完成")

## 5. 基础测试

In [ ]:
# 参数设置
seq_len = 8
d_k = 64  # 键/查询的维度
d_v = 64  # 值的维度

# 生成随机的Q, K, V矩阵
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_v)

# 创建注意力层
attention = ScaledDotProductAttention()

# 前向传播
output, weights = attention.forward(Q, K, V)

print(f"查询矩阵Q形状: {Q.shape}")
print(f"键矩阵K形状: {K.shape}")
print(f"值矩阵V形状: {V.shape}")
print(f"\n输出矩阵形状: {output.shape}")
print(f"注意力权重形状: {weights.shape}")
print(f"\n每行权重之和: {weights.sum(axis=-1)}")

## 6. 可视化注意力权重

In [ ]:
# 创建热力图
plt.figure(figsize=(10, 8))
sns.heatmap(weights, annot=True, fmt='.3f', cmap='YlOrRd', 
            xticklabels=range(seq_len), yticklabels=range(seq_len),
            cbar_kws={'label': '注意力权重'})
plt.xlabel('键位置 (Key Position)', fontsize=12)
plt.ylabel('查询位置 (Query Position)', fontsize=12)
plt.title('Scaled Dot-Product Attention 权重矩阵', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

print("注意：每一行表示一个查询位置对所有键位置的注意力分布")

## 7. Padding Mask演示

In [ ]:
def create_padding_mask(seq_len, padding_positions):
    """创建padding mask"""
    mask = np.zeros((seq_len, seq_len), dtype=bool)
    for pos in padding_positions:
        mask[:, pos] = True
    return mask

# 假设序列长度为6，最后2个位置是padding
seq_len = 6
padding_positions = [4, 5]

Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_v)

# 创建padding mask
padding_mask = create_padding_mask(seq_len, padding_positions)

# 应用mask
output_masked, weights_masked = scaled_dot_product_attention(Q, K, V, mask=padding_mask)

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Mask矩阵
sns.heatmap(padding_mask.astype(int), annot=True, fmt='d', cmap='Greys', 
            ax=ax1, cbar=False, xticklabels=range(seq_len), yticklabels=range(seq_len))
ax1.set_title('Padding Mask (黑色=mask)', fontsize=12)
ax1.set_xlabel('键位置')
ax1.set_ylabel('查询位置')

# 注意力权重
sns.heatmap(weights_masked, annot=True, fmt='.3f', cmap='YlOrRd', 
            ax=ax2, xticklabels=range(seq_len), yticklabels=range(seq_len))
ax2.set_title('应用Mask后的注意力权重', fontsize=12)
ax2.set_xlabel('键位置')
ax2.set_ylabel('查询位置')

plt.tight_layout()
plt.show()

print(f"注意：位置{padding_positions}的注意力权重接近0")

## 8. Causal Mask演示（用于Decoder）

In [ ]:
def create_causal_mask(seq_len):
    """创建因果mask，防止看到未来信息"""
    mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
    return mask

# 创建一个简单的序列
seq_len = 5
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_v)

# 创建因果mask
causal_mask = create_causal_mask(seq_len)

# 应用因果mask
output_causal, weights_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Causal Mask
sns.heatmap(causal_mask.astype(int), annot=True, fmt='d', cmap='Greys', 
            ax=ax1, cbar=False, xticklabels=range(seq_len), yticklabels=range(seq_len))
ax1.set_title('Causal Mask (上三角=mask)', fontsize=12)
ax1.set_xlabel('键位置')
ax1.set_ylabel('查询位置')

# 注意力权重
sns.heatmap(weights_causal, annot=True, fmt='.3f', cmap='YlOrRd', 
            ax=ax2, xticklabels=range(seq_len), yticklabels=range(seq_len))
ax2.set_title('应用Causal Mask后的注意力权重', fontsize=12)
ax2.set_xlabel('键位置')
ax2.set_ylabel('查询位置')

plt.tight_layout()
plt.show()

print("注意：每个查询位置只能attend到当前及之前的键位置")
print("这确保了在生成序列时不会看到未来的信息")

## 9. 缩放的重要性演示

In [ ]:
# 使用不同的d_k值来展示缩放的效果
d_k_values = [16, 64, 256, 512]
seq_len = 5

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, d_k in enumerate(d_k_values):
    Q = np.random.randn(seq_len, d_k)
    K = np.random.randn(seq_len, d_k)
    V = np.random.randn(seq_len, d_k)
    
    # 不缩放
    scores_no_scale = np.matmul(Q, np.swapaxes(K, -2, -1))
    weights_no_scale = softmax(scores_no_scale, axis=-1)
    
    # 缩放
    scores_scaled = scores_no_scale / np.sqrt(d_k)
    weights_scaled = softmax(scores_scaled, axis=-1)
    
    # 计算权重差异
    diff = np.abs(weights_no_scale - weights_scaled)
    
    # 绘制热力图
    sns.heatmap(diff, annot=True, fmt='.3f', cmap='RdYlGn_r', ax=axes[idx],
                xticklabels=range(seq_len), yticklabels=range(seq_len),
                vmin=0, vmax=0.3)
    axes[idx].set_title(f'd_k={d_k}\n权重差异 (不缩放 vs 缩放)', fontsize=11)
    axes[idx].set_xlabel('键位置')
    axes[idx].set_ylabel('查询位置')
    
    # 添加统计信息
    max_weight_no_scale = weights_no_scale.max()
    max_weight_scaled = weights_scaled.max()
    axes[idx].text(0.5, -0.15, 
                   f'最大权重: 不缩放={max_weight_no_scale:.3f}, 缩放={max_weight_scaled:.3f}',
                   transform=axes[idx].transAxes, ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print("观察：")
print("- d_k越大，不缩放时权重越集中（最大值接近1）")
print("- 缩放后，权重分布更加平滑和均匀")
print("- 这有助于避免梯度消失，使训练更稳定")

## 10. 批量处理演示

In [ ]:
# 批量处理
batch_size = 3
seq_len = 4
d_k = 32
d_v = 32

Q_batch = np.random.randn(batch_size, seq_len, d_k)
K_batch = np.random.randn(batch_size, seq_len, d_k)
V_batch = np.random.randn(batch_size, seq_len, d_v)

# 使用简化函数
output_batch, weights_batch = scaled_dot_product_attention(Q_batch, K_batch, V_batch)

print(f"批量查询矩阵Q形状: {Q_batch.shape}")
print(f"批量输出矩阵形状: {output_batch.shape}")
print(f"批量注意力权重形状: {weights_batch.shape}")

# 可视化每个batch的注意力权重
fig, axes = plt.subplots(1, batch_size, figsize=(15, 4))

for i in range(batch_size):
    sns.heatmap(weights_batch[i], annot=True, fmt='.2f', cmap='YlOrRd', 
                ax=axes[i], xticklabels=range(seq_len), yticklabels=range(seq_len))
    axes[i].set_title(f'Batch {i+1}', fontsize=12)
    axes[i].set_xlabel('键位置')
    if i == 0:
        axes[i].set_ylabel('查询位置')

plt.tight_layout()
plt.show()

## 11. 实际应用示例：序列编码

In [ ]:
# 模拟一个简单的文本序列编码任务
# 假设我们有一个句子: "Hello world from AI"
words = ["Hello", "world", "from", "AI"]
seq_len = len(words)
d_model = 64

# 随机初始化词嵌入（实际应用中会使用预训练的词向量）
word_embeddings = np.random.randn(seq_len, d_model)

# 在自注意力中，Q, K, V都来自同一个输入
Q = word_embeddings
K = word_embeddings
V = word_embeddings

# 计算自注意力
output, weights = scaled_dot_product_attention(Q, K, V)

# 可视化注意力权重
plt.figure(figsize=(8, 6))
sns.heatmap(weights, annot=True, fmt='.3f', cmap='Blues', 
            xticklabels=words, yticklabels=words, cbar_kws={'label': '注意力权重'})
plt.xlabel('键 (Key)', fontsize=12)
plt.ylabel('查询 (Query)', fontsize=12)
plt.title('Self-Attention 权重矩阵\n(模拟句子: "Hello world from AI")', fontsize=13, pad=15)
plt.tight_layout()
plt.show()

print("解释：")
print("- 每一行显示一个词对所有词（包括自己）的注意力分布")
print("- 自注意力允许每个词关注句子中的所有位置")
print("- 这是Transformer编码器的核心机制")

## 12. 总结

### 关键要点

1. **缩放的必要性**：
   - 当$d_k$较大时，点积值会很大
   - 导致softmax进入饱和区，梯度接近0
   - 除以$\sqrt{d_k}$保持方差稳定

2. **Mask机制**：
   - **Padding Mask**：处理变长序列，忽略padding位置
   - **Causal Mask**：用于decoder，防止看到未来信息

3. **计算效率**：
   - 使用矩阵乘法，可以高度并行化
   - 支持批量处理
   - 是Multi-Head Attention的基础

4. **应用场景**：
   - Transformer的Encoder和Decoder
   - BERT、GPT等预训练模型
   - 各种基于注意力的序列模型

### 与其他注意力机制的区别

| 特性 | Soft Attention | Scaled Dot-Product |
|------|---------------|--------------------|
| 相似度计算 | 可学习的网络 | 点积 |
| 缩放因子 | 无 | $\sqrt{d_k}$ |
| 计算效率 | 较慢 | 快（矩阵运算） |
| 参数量 | 有可学习参数 | 无额外参数 |
| 应用 | Seq2seq | Transformer |